# 01. Tokenization

## 학습 목표
- 토큰화가 왜 필요한지, 어휘 크기 vs 시퀀스 길이 트레이드오프 이해
- BPE, WordPiece, SentencePiece 알고리즘의 핵심 원리 파악
- BPE를 처음부터 직접 구현
- 한국어 토큰화 비교 실험 (GPT vs BERT)

## 참고 자료
- [Andrej Karpathy - Let's build the GPT Tokenizer](https://www.youtube.com/watch?v=zduSFxRajkE)
- [HuggingFace Tokenizers 문서](https://huggingface.co/docs/tokenizers)

---

In [ ]:
# Google Colab에서 필요한 패키지 설치
# !pip install transformers tiktoken sentencepiece matplotlib -q

In [ ]:
import re
from collections import Counter, defaultdict
import matplotlib.pyplot as plt
import matplotlib

# 한글 폰트 설정 (Colab 환경)
matplotlib.rcParams['font.family'] = 'DejaVu Sans'
matplotlib.rcParams['axes.unicode_minus'] = False

## 1. 왜 Tokenization이 중요한가

LLM은 텍스트를 직접 이해하지 못한다. 모델에 넣기 전에 텍스트를 **숫자(Token ID)** 로 변환해야 한다.

```
"안녕하세요" -> [30903, 233, 31543, ...] -> Embedding -> Model
```

### 어휘 크기(Vocabulary Size) vs 시퀀스 길이(Sequence Length) 트레이드오프

| 전략 | 어휘 크기 | 시퀀스 길이 | 장점 | 단점 |
|------|----------|----------|------|------|
| 문자 단위 | 아주 작음 (~300) | 매우 김 | OOV 없음 | 의미 파악 어려움, 계산 비용 높음 |
| 단어 단위 | 매우 큼 (~100K+) | 짧음 | 의미 보존 | OOV 문제, Embedding 테이블 너무 큼 |
| **서브워드** | **적절 (~32K-100K)** | **적절** | **OOV 최소 + 의미 보존** | **알고리즘 필요** |

**핵심**: 어휘 크기가 커지면 Embedding 테이블 크기가 $V \times d_{\text{model}}$ 로 커지고,
시퀀스가 길어지면 Attention 계산이 $O(n^2)$으로 느려진다.

서브워드 토큰화는 이 둘 사이의 최적 균형을 찾는 방법이다.

In [ ]:
# 문자 단위 vs 단어 단위 vs 서브워드 비교
text = "I love natural language processing"

# 문자 단위
char_tokens = list(text)
print(f"문자 단위: {char_tokens}")
print(f"  토큰 수: {len(char_tokens)}, 어휘 크기: {len(set(char_tokens))}\n")

# 단어 단위
word_tokens = text.split()
print(f"단어 단위: {word_tokens}")
print(f"  토큰 수: {len(word_tokens)}, 어휘 크기: {len(set(word_tokens))}\n")

# 서브워드 (예시 - BPE 결과)
subword_tokens = ["I", " love", " natural", " language", " process", "ing"]
print(f"서브워드: {subword_tokens}")
print(f"  토큰 수: {len(subword_tokens)}")
print(f"  -> 'processing'을 'process' + 'ing'으로 분리: 의미 보존 + 어휘 효율적")

---
## 2. 문자 단위 vs 단어 단위 vs 서브워드 비교

### 한국어에서의 차이

한국어는 **교착어**로 조사, 어미가 붙어 단어 경계가 모호하다.
- "나는" = "나" + "는"(조사)
- "먹었다" = "먹" + "었" + "다"(어미)

이 때문에 단어 단위 토큰화는 한국어에 적합하지 않고, 서브워드가 더 효과적이다.

In [ ]:
# 한국어 예시로 세 가지 접근법 비교
text_ko = "강남에서 분위기 좋은 카페 추천해줘"

# 문자 단위
char_tokens_ko = list(text_ko.replace(" ", ""))
print(f"문자 단위: {char_tokens_ko}")
print(f"  토큰 수: {len(char_tokens_ko)}")
print(f"  문제: 각 글자만으로는 의미를 알기 어려움\n")

# 단어 단위 (띄어쓰기 기준)
word_tokens_ko = text_ko.split()
print(f"단어 단위: {word_tokens_ko}")
print(f"  토큰 수: {len(word_tokens_ko)}")
print(f"  문제: '강남에서', '강남으로', '강남은'이 전부 다른 토큰\n")

# 서브워드 (예시)
subword_ko = ["강남", "에서", " 분위기", " 좋", "은", " 카페", " 추천", "해", "줘"]
print(f"서브워드: {subword_ko}")
print(f"  토큰 수: {len(subword_ko)}")
print(f"  장점: '강남'을 공유하면서도 조사를 분리")

---
## 3. BPE (Byte Pair Encoding) 직접 구현

BPE는 GPT 시리즈에서 사용하는 토큰화 알고리즘이다.

### 알고리즘
1. 모든 문자를 개별 토큰으로 시작
2. 가장 많이 인접하는 토큰 쌍을 찾음
3. 그 쌍을 하나로 병합 (merge)
4. 원하는 어휘 크기가 될 때까지 2-3 반복

$$\text{Score}(a, b) = \text{count}(a, b)$$

핵심: **빈도 기반**으로 바텀업 학습. 자주 등장하는 문자 조합이 하나의 토큰이 된다.

In [ ]:
def get_pair_counts(vocab):
    """어휘에서 인접 토큰 쌍의 빈도를 계산"""
    pairs = Counter()
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pairs[(symbols[i], symbols[i + 1])] += freq
    return pairs


def merge_pair(pair, vocab):
    """어휘에서 해당 pair를 병합"""
    new_vocab = {}
    bigram = ' '.join(pair)
    replacement = ''.join(pair)
    for word, freq in vocab.items():
        # 정확한 병합을 위해 정규식 사용
        new_word = word.replace(bigram, replacement)
        new_vocab[new_word] = freq
    return new_vocab


# 학습 데이터에서 단어 빈도 추출 (단어 끝에 </w> 추가)
corpus = "low low low low low lower lower newest newest newest newest newest newest widest widest widest"

# Step 0: 각 단어를 문자 단위로 분리
word_freq = Counter(corpus.split())
print(f"단어 빈도: {dict(word_freq)}\n")

# 문자 단위로 분리 (</w>는 단어 끝 표시)
vocab = {}
for word, freq in word_freq.items():
    chars = ' '.join(list(word)) + ' </w>'
    vocab[chars] = freq

print(f"초기 어휘:")
for word, freq in vocab.items():
    print(f"  {word} : {freq}")

In [ ]:
# BPE 학습: step-by-step merge 과정
num_merges = 10
merge_history = []

print("=" * 60)
print("BPE Merge 과정")
print("=" * 60)

for i in range(num_merges):
    pairs = get_pair_counts(vocab)
    if not pairs:
        break

    # 가장 빈도 높은 pair
    best_pair = max(pairs, key=pairs.get)
    best_count = pairs[best_pair]

    merge_history.append((best_pair, best_count))
    vocab = merge_pair(best_pair, vocab)

    print(f"\nStep {i + 1}: merge '{best_pair[0]}' + '{best_pair[1]}' (count: {best_count})")
    print(f"  -> '{best_pair[0]}{best_pair[1]}'")
    print(f"  현재 어휘:")
    for word, freq in vocab.items():
        print(f"    {word} : {freq}")

In [ ]:
# Merge 히스토리 시각화
fig, ax = plt.subplots(figsize=(10, 5))

steps = list(range(1, len(merge_history) + 1))
labels = [f"'{p[0]}'+'{p[1]}'" for p, _ in merge_history]
counts = [c for _, c in merge_history]

bars = ax.bar(steps, counts, color='steelblue', edgecolor='white')
ax.set_xlabel('Merge Step')
ax.set_ylabel('Pair Frequency')
ax.set_title('BPE Merge History: Pair Frequency at Each Step')
ax.set_xticks(steps)

# 바 위에 레이블 표시
for bar, label in zip(bars, labels):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.2,
            label, ha='center', va='bottom', fontsize=8, rotation=45)

plt.tight_layout()
plt.show()

### BPE 토큰화 적용 (Encoding)

학습된 merge 규칙을 순서대로 새로운 텍스트에 적용한다.

In [ ]:
def bpe_encode(word, merge_rules):
    """학습된 merge rule을 적용하여 단어를 토큰화"""
    symbols = list(word) + ['</w>']

    for pair, _ in merge_rules:
        i = 0
        while i < len(symbols) - 1:
            if symbols[i] == pair[0] and symbols[i + 1] == pair[1]:
                symbols = symbols[:i] + [pair[0] + pair[1]] + symbols[i + 2:]
            else:
                i += 1
    return symbols


# 학습 데이터에 있는 단어
test_words = ["low", "lower", "newest", "widest"]
print("학습 데이터 단어:")
for word in test_words:
    tokens = bpe_encode(word, merge_history)
    print(f"  '{word}' -> {tokens}")

# 학습 데이터에 없는 단어 (미등록어 처리)
print(f"\n미등록 단어:")
unseen = ["lowest", "newer", "high"]
for word in unseen:
    tokens = bpe_encode(word, merge_history)
    print(f"  '{word}' -> {tokens}")
    print(f"    -> 학습된 서브워드로 분해되므로 OOV 없음!")

---
## 4. WordPiece: BERT에서 사용

WordPiece는 Google이 BERT에서 사용한 토큰화 알고리즘이다.

### BPE vs WordPiece 핵심 차이

| | BPE | WordPiece |
|---|-----|----------|
| 병합 기준 | 가장 **빈도 높은** 쌍 | 가장 **우도(likelihood)를 높이는** 쌍 |
| 점수 | $\text{count}(ab)$ | $\frac{\text{count}(ab)}{\text{count}(a) \times \text{count}(b)}$ |
| 특징 | 단순하고 직관적 | 희귀 조합도 병합 가능 |
| 서브워드 표시 | 없음 | `##` 접두사 |
| 대표 모델 | GPT 시리즈 | BERT, DistilBERT |

WordPiece 점수: 두 토큰이 각각 자주 나오는데도 **같이** 나오는 경우를 더 높이 평가.

In [ ]:
# WordPiece 점수 vs BPE 점수 비교 예시
# 가상의 corpus 빈도
token_counts = {'e': 100, 's': 80, 't': 90, 'es': 50, 'st': 30}

# BPE 점수: 단순 빈도
bpe_score_es = token_counts['es']
bpe_score_st = token_counts['st']

# WordPiece 점수: 빈도 / (각각의 빈도 곱)
wp_score_es = token_counts['es'] / (token_counts['e'] * token_counts['s'])
wp_score_st = token_counts['st'] / (token_counts['s'] * token_counts['t'])

print("BPE 점수 (단순 빈도):")
print(f"  score('e','s') = {bpe_score_es}")
print(f"  score('s','t') = {bpe_score_st}")
print(f"  -> BPE는 'es'를 먼저 병합 (빈도 높으니까)\n")

print("WordPiece 점수 (count(ab) / count(a)*count(b)):")
print(f"  score('e','s') = {token_counts['es']} / ({token_counts['e']}*{token_counts['s']}) = {wp_score_es:.5f}")
print(f"  score('s','t') = {token_counts['st']} / ({token_counts['s']}*{token_counts['t']}) = {wp_score_st:.5f}")
print(f"  -> WordPiece도 'es'를 먼저 병합 (확률 기준으로 더 의미있는 조합)")

In [ ]:
# BERT WordPiece 토큰화 예시 (직접 시뮬레이션)
# 실제 BERT는 ## 접두사로 서브워드를 표시

def wordpiece_tokenize_greedy(word, vocab_set):
    """간단한 WordPiece 토큰화 (Greedy longest-match)"""
    tokens = []
    start = 0
    while start < len(word):
        end = len(word)
        found = False
        while start < end:
            substr = word[start:end]
            if start > 0:
                substr = '##' + substr
            if substr in vocab_set:
                tokens.append(substr)
                found = True
                break
            end -= 1
        if not found:
            tokens.append('[UNK]')
            start += 1
        else:
            start = end
    return tokens


# 가상의 WordPiece 어휘
vocab_set = {'un', '##break', '##able', '##ing', 'play', '##ful',
             'the', 'cat', 'sat', '##s', 'help', '##ed', '##less'}

test_words = ['unbreakable', 'playing', 'helpful', 'helpless', 'cats']

print("WordPiece Tokenization (가상 어휘):")
print(f"  어휘: {vocab_set}\n")
for word in test_words:
    tokens = wordpiece_tokenize_greedy(word, vocab_set)
    print(f"  '{word}' -> {tokens}")

print(f"\n특징: ## 접두사가 단어 중간/끝 위치의 서브워드를 표시")

---
## 5. SentencePiece: 언어 독립적 토큰화

### BPE/WordPiece의 문제점
- 띄어쓰기로 단어를 구분 -> 한국어, 중국어, 일본어 등에 부적합
- 언어별 전처리(tokenizer) 필요

### SentencePiece의 해결책
- **원문(raw text)을 직접** 토큰화 (띄어쓰기 무시)
- 공백을 특수 문자 `▁` (U+2581)로 치환하여 처리
- BPE 또는 **Unigram** 모델 선택 가능
- LLaMA, T5, ALBERT 등에서 사용

### Unigram Model (SentencePiece 기본)
- BPE와 반대: **큰 어휘에서 시작해서 줄여나감**
- 각 서브워드의 확률을 기반으로, 전체 우도를 최대화하는 분할을 선택

$$x^* = \arg\max_{x \in S(x)} \prod_{i=1}^{n} P(x_i)$$

In [ ]:
# BPE vs Unigram 방향 비교
print("BPE (Bottom-Up):")
print("  시작: ['h', 'e', 'l', 'l', 'o']")
print("  step1: ['h', 'e', 'll', 'o']      <- 'l'+'l' 병합")
print("  step2: ['h', 'e', 'llo']           <- 'll'+'o' 병합")
print("  step3: ['h', 'ello']               <- 'e'+'llo' 병합")
print("  step4: ['hello']                   <- 'h'+'ello' 병합")
print()
print("Unigram (Top-Down):")
print("  시작: ['hello', 'hell', 'ello', 'hel', 'ell', 'llo', 'he', 'll', 'lo', 'h', 'e', 'l', 'o']")
print("  step1: 각 서브워드의 확률 계산")
print("  step2: 우도를 가장 적게 감소시키는 서브워드 제거")
print("  step3: 원하는 어휘 크기까지 반복")

---
## 6. 한국어 토큰화 비교 실험

"강남에서 분위기 좋은 카페 추천해줘"를 GPT tokenizer vs BERT tokenizer로 비교해보자.

- GPT: BPE 기반 (tiktoken / cl100k_base)
- BERT: WordPiece 기반 (bert-base-multilingual-cased)

In [ ]:
# 실제 토큰화 비교
# 필요한 패키지: pip install tiktoken transformers

import tiktoken
from transformers import AutoTokenizer

text_ko = "강남에서 분위기 좋은 카페 추천해줘"

# GPT-4 tokenizer (BPE - cl100k_base)
enc_gpt = tiktoken.get_encoding("cl100k_base")
gpt_tokens = enc_gpt.encode(text_ko)
gpt_decoded = [enc_gpt.decode([t]) for t in gpt_tokens]

print("=" * 60)
print(f"원문: {text_ko}")
print("=" * 60)

print(f"\nGPT (BPE, cl100k_base):")
print(f"  Token IDs: {gpt_tokens}")
print(f"  토큰 수:  {len(gpt_tokens)}")
print(f"  분해:     {gpt_decoded}")

# BERT tokenizer (WordPiece)
bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")
bert_tokens = bert_tokenizer.tokenize(text_ko)
bert_ids = bert_tokenizer.encode(text_ko, add_special_tokens=False)

print(f"\nBERT (WordPiece, multilingual):")
print(f"  Token IDs: {bert_ids}")
print(f"  토큰 수:  {len(bert_ids)}")
print(f"  분해:     {bert_tokens}")

In [ ]:
# 토큰화 결과 시각화
fig, axes = plt.subplots(2, 1, figsize=(14, 6))

# GPT 토큰화
ax = axes[0]
colors_gpt = plt.cm.Set3([i / len(gpt_decoded) for i in range(len(gpt_decoded))])
for i, (token, color) in enumerate(zip(gpt_decoded, colors_gpt)):
    ax.barh(0, 1, left=i, color=color, edgecolor='black', linewidth=0.5)
    display_text = repr(token)[1:-1]  # 공백 등을 보이게
    ax.text(i + 0.5, 0, display_text, ha='center', va='center', fontsize=9)
ax.set_xlim(0, max(len(gpt_decoded), len(bert_tokens)))
ax.set_ylim(-0.5, 0.5)
ax.set_yticks([])
ax.set_title(f'GPT (BPE, cl100k_base) - {len(gpt_decoded)} tokens')

# BERT 토큰화
ax = axes[1]
colors_bert = plt.cm.Set3([i / len(bert_tokens) for i in range(len(bert_tokens))])
for i, (token, color) in enumerate(zip(bert_tokens, colors_bert)):
    ax.barh(0, 1, left=i, color=color, edgecolor='black', linewidth=0.5)
    ax.text(i + 0.5, 0, token, ha='center', va='center', fontsize=9)
ax.set_xlim(0, max(len(gpt_decoded), len(bert_tokens)))
ax.set_ylim(-0.5, 0.5)
ax.set_yticks([])
ax.set_title(f'BERT (WordPiece, multilingual) - {len(bert_tokens)} tokens')

plt.suptitle(f'Tokenization: "{text_ko}"', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# 의미 보존 분석
print("=" * 60)
print("토큰화 차이가 의미 보존에 미치는 영향")
print("=" * 60)

# 여러 한국어 문장으로 비교
test_sentences = [
    "강남에서 분위기 좋은 카페 추천해줘",
    "오늘 날씨가 정말 좋네요",
    "인공지능이 세상을 바꾸고 있습니다",
    "Transformer 모델의 Self-Attention 메커니즘",
]

print(f"{'\uc6d0문':<40} {'GPT 토큰수':>10} {'BERT 토큰수':>10} {'토큰수 차이':>10}")
print("-" * 75)

gpt_total = 0
bert_total = 0

for sent in test_sentences:
    g_count = len(enc_gpt.encode(sent))
    b_count = len(bert_tokenizer.encode(sent, add_special_tokens=False))
    gpt_total += g_count
    bert_total += b_count
    print(f"{sent:<40} {g_count:>10} {b_count:>10} {b_count - g_count:>+10}")

print("-" * 75)
print(f"{'\ud569\uacc4':<40} {gpt_total:>10} {bert_total:>10} {bert_total - gpt_total:>+10}")
print()
print("분석:")
print("  - GPT(BPE)는 한국어를 바이트 수준에서 처리 -> 토큰 수가 많을 수 있음")
print("  - BERT multilingual은 한국어 문자/음절 단위 어휘 보유")
print("  - 토큰 수가 많을수록 -> 시퀀스 길이 증가 -> 계산 비용 증가")
print("  - 토큰 수가 많을수록 -> 의미 단위가 쪼개져 모델이 의미 파악하기 어려움")

In [ ]:
# 토큰 수 비교 시각화
fig, ax = plt.subplots(figsize=(10, 5))

labels = [s[:15] + '...' if len(s) > 15 else s for s in test_sentences]
gpt_counts = [len(enc_gpt.encode(s)) for s in test_sentences]
bert_counts = [len(bert_tokenizer.encode(s, add_special_tokens=False)) for s in test_sentences]

x = range(len(test_sentences))
width = 0.35

bars1 = ax.bar([i - width/2 for i in x], gpt_counts, width, label='GPT (BPE)', color='steelblue')
bars2 = ax.bar([i + width/2 for i in x], bert_counts, width, label='BERT (WordPiece)', color='coral')

ax.set_ylabel('Token Count')
ax.set_title('Korean Text: GPT vs BERT Token Count')
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=15, ha='right')
ax.legend()

# 바 위에 숫자 표시
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
            str(int(bar.get_height())), ha='center', va='bottom', fontsize=10)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
            str(int(bar.get_height())), ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

---
## 연습 문제

아래 문제를 직접 풀어보세요.

### 연습 1: BPE merge를 더 많이 수행하면 어떻게 될까?

아래 corpus로 BPE를 15번 merge하고, 최종 어휘를 확인하세요.
그리고 `"lowest"` 단어를 토큰화해보세요.

In [ ]:
corpus_ex = "low low low lower lower lowest lowest newest newest newest widest widest higher higher highest"

# TODO: 위 corpus로 BPE를 15번 merge 수행하세요
# 1. word_freq 을 구하고
# 2. 문자 단위로 분리한 초기 vocab을 만들고
# 3. 15번 merge를 수행하고 최종 vocab을 출력하고
# 4. bpe_encode("lowest", merge_history)를 호출하세요


### 연습 2: 다양한 한국어 문장으로 토큰화 비교

아래 한국어 문장들을 GPT와 BERT 토큰화로 비교하고,
어떤 문장에서 두 토큰화의 차이가 가장 큰지 분석하세요.

In [ ]:
sentences = [
    "서울 지하철 2호선 신당역에서 내리세요",
    "딥러닝 모델의 역전파 알고리즘",
    "ChatGPT는 OpenAI가 만든 대화형 AI입니다",
    "아버지가 방에 들어가신다",
    "프로그래밍 언어 Python으로 데이터 분석하기",
]

# TODO: 각 문장을 GPT/BERT로 토큰화하고
# 1. 토큰 수를 비교하는 테이블을 출력하세요
# 2. 토큰 수 차이가 가장 큰 문장의 실제 토큰을 출력하세요
# 3. 왜 차이가 큰지 분석하세요


---
## 핵심 정리

| 개념 | 설명 | 대표 모델 |
|------|------|----------|
| BPE | 빈도 기반 바텀업 병합 | GPT-2, GPT-3, GPT-4 |
| WordPiece | 우도 기반 병합, ## 접두사 | BERT, DistilBERT |
| SentencePiece | 언어 독립적, Unigram/BPE | LLaMA, T5, ALBERT |
| 어휘 크기 | 클수록 Embedding 테이블 큼 | 보통 32K~100K |
| 시퀀스 길이 | 길수록 Attention 계산 비용 $O(n^2)$ | - |
| 한국어 | 교착어 특성으로 서브워드가 필수 | - |

**다음 노트북**: [02-scaling-laws.ipynb](02-scaling-laws.ipynb) - Scaling Laws